# VanniKawachh — End-to-End External Voice Distress Benchmark

This Colab evaluates the **complete fixed `SV-1411/drone` voice-distress pipeline as a black box** on an external benchmark. It does **not retrain** the model.

## Goal
- Genuine distress → `DISTRESS`
- Excited shouting/cheering/laughter/singing/ordinary shouting → `NON-DISTRESS`
- Report accuracy, balanced accuracy, precision, recall, Macro-F1, specificity, false-positive rate, and the false-positive files.

**Important:** an external speech/emotion benchmark is not proof of zero real-world false positives. Zero FP here means zero observed false positives on the tested benchmark.

In [ ]:
import os, sys, subprocess, pathlib, json, hashlib
REPO='/content/drone'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/SV-1411/drone.git',REPO],check=True)
os.chdir(REPO)
print('Repo:',REPO)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
import subprocess, sys
pkgs=['numpy>=1.26','scikit-learn','pandas','matplotlib','tensorflow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs,check=True)
print('Dependencies installed.')

## Upload the trained model

Upload the trained `distress_svm.pkl` from your model artifacts. If you have `distress_model_meta.pkl`, upload that too. The repository source expects the trained model under `hub/models/`.

In [ ]:
from google.colab import files
from pathlib import Path
MODEL_DIR=Path(REPO)/'hub'/'models'
MODEL_DIR.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.pkl'):
        Path('/content',name).replace(MODEL_DIR/name)
print('Model artifacts:',[p.name for p in MODEL_DIR.glob('*.pkl')])
if not (MODEL_DIR/'distress_svm.pkl').exists():
    raise FileNotFoundError('Upload distress_svm.pkl before continuing.')

## External benchmark input

Prepare a ZIP containing:

```text
external_benchmark/
  labels.csv
  audio/
    sample001.wav
    sample002.wav
    ...
```

`labels.csv` must contain `path,label`.

Example:

```csv
path,label
audio/sample001.wav,distress
audio/sample002.wav,excited
audio/sample003.wav,laughter
```

Use a real external benchmark such as H-VB when available. IEMOCAP/MSP-Podcast can be supplementary external validation, but they are emotion datasets rather than dedicated emergency-distress datasets.

In [ ]:
from google.colab import files
import zipfile, shutil
BENCH=Path('/content/external_benchmark')
if BENCH.exists(): shutil.rmtree(BENCH)
BENCH.mkdir()
z=files.upload()
zips=[n for n in z if n.lower().endswith('.zip')]
if not zips: raise ValueError('Upload a ZIP containing labels.csv and WAV files.')
with zipfile.ZipFile('/content/'+zips[0]) as zz: zz.extractall(BENCH)
print('Extracted:',BENCH)

In [ ]:
import pandas as pd, numpy as np
label_files=list(BENCH.rglob('labels.csv'))
if not label_files: raise FileNotFoundError('No labels.csv found.')
LABELS=label_files[0]
df=pd.read_csv(LABELS)
if not {'path','label'}.issubset(df.columns): raise ValueError('labels.csv requires path,label columns.')
def resolve_path(x):
    p=Path(str(x))
    return str(p if p.is_absolute() else (LABELS.parent/p).resolve())
df['path']=df['path'].map(resolve_path)
missing=df[~df.path.map(os.path.exists)]
if len(missing): raise FileNotFoundError(f'{len(missing)} audio paths do not exist; first: {missing.path.iloc[0]}')
print(df.label.value_counts())
display(df.head())

## Configure the ground-truth mapping

The mapping is intentionally conservative. **Do not map excitement, cheering, laughter, singing, or ordinary shouting to distress.** If the external dataset has a directly annotated `distress` class, map that class to `distress`.

In [ ]:
DISTRESS_LABELS={'distress','panic','scream_distress','pain','cry_distress'}
def map_truth(label):
    s=str(label).strip().lower().replace(' ','_')
    return 'distress' if s in DISTRESS_LABELS else 'non_distress'
df['truth']=df['label'].map(map_truth)
print(df['truth'].value_counts())
if df.truth.nunique()!=2: raise ValueError('After mapping, benchmark must contain both distress and non_distress.')

## Run the complete repository inference pipeline

This cell calls the repository's existing YAMNet + feature + `DistressClassifier` path. It does not fit or retrain anything.

In [ ]:
import sys, wave, traceback
sys.path.insert(0,REPO)
from hub.distress_classifier import DistressClassifier, build_feature_vector
from hub.yamnet_detector import get_detector
detector=get_detector()
if detector is None: raise RuntimeError('YAMNet detector could not be initialized. Check hub/models/yamnet.tflite and dependencies.')
model=DistressClassifier()
def load_wav(path):
    with wave.open(path,'rb') as wf:
        sr=wf.getframerate(); ch=wf.getnchannels(); width=wf.getsampwidth(); raw=wf.readframes(wf.getnframes())
    if width==2: x=np.frombuffer(raw,dtype='<i2').astype(np.float32)/32768.0
    elif width==1: x=(np.frombuffer(raw,dtype=np.uint8).astype(np.float32)-128)/128.0
    else: raise ValueError(f'Unsupported WAV sample width {width}: {width*8}-bit')
    if ch>1: x=x.reshape(-1,ch).mean(axis=1)
    return x.astype(np.float32),int(sr)
rows=[]
for i,r in df.iterrows():
    try:
        audio,sr=load_wav(r.path)
        rep=detector.embedding(audio,sr)
        if rep is None: rep=detector.class_score_vector(audio,sr)
        pred=model.predict_features(build_feature_vector(audio,sr,rep))
        rows.append({**r.to_dict(),'prediction':pred.predicted_class,'distress_probability':float(pred.distress_probability),'error':''})
    except Exception as e:
        rows.append({**r.to_dict(),'prediction':'ERROR','distress_probability':np.nan,'error':repr(e)})
results=pd.DataFrame(rows)
print('Processed:',len(results),'Errors:',int((results.prediction=='ERROR').sum()))
display(results.head())

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
valid=results[results.prediction!='ERROR'].copy()
valid['pred_binary']=np.where(valid.prediction.astype(str).eq('distress'),'distress','non_distress')
y=valid.truth; p=valid.pred_binary
acc=accuracy_score(y,p); bal=balanced_accuracy_score(y,p)
prec=precision_score(y,p,pos_label='distress',zero_division=0)
rec=recall_score(y,p,pos_label='distress',zero_division=0)
f1=f1_score(y,p,pos_label='distress',zero_division=0)
tn,fp,fn,tp=confusion_matrix(y,p,labels=['non_distress','distress']).ravel()
spec=tn/(tn+fp) if tn+fp else 0.0; fpr=fp/(fp+tn) if fp+tn else 0.0
summary={'samples':len(valid),'accuracy':acc,'balanced_accuracy':bal,'distress_precision':prec,'distress_recall':rec,'macro_f1':f1,'specificity':spec,'false_positive_rate':fpr,'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'zero_false_positives':bool(fp==0)}
print('=== END-TO-END EXTERNAL BENCHMARK ===')
for k,v in summary.items(): print(f'{k}: {v:.4f}' if isinstance(v,float) else f'{k}: {v}')
print('\nClassification report:\n',classification_report(y,p,labels=['non_distress','distress'],zero_division=0))

In [ ]:
fps=valid[(valid.truth=='non_distress')&(valid.pred_binary=='distress')].sort_values('distress_probability',ascending=False)
print(f'False positives: {len(fps)} / {int((valid.truth=="non_distress").sum())} non-distress samples')
display(fps[['path','label','distress_probability']].head(100))
if len(fps)==0: print('ZERO-FP RESULT on this benchmark.')
else: print('ZERO-FP REQUIREMENT FAILED on this benchmark.')

In [ ]:
import matplotlib.pyplot as plt
cm=confusion_matrix(y,p,labels=['non_distress','distress'])
fig,ax=plt.subplots(figsize=(5,4)); ax.imshow(cm)
ax.set_xticks([0,1],['Non-distress','Distress']); ax.set_yticks([0,1],['Non-distress','Distress'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('End-to-End Confusion Matrix')
for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center')
plt.tight_layout(); plt.savefig('/content/confusion_matrix.png',dpi=160); plt.show()

In [ ]:
results.to_csv('/content/external_benchmark_results.csv',index=False)
with open('/content/external_benchmark_summary.json','w') as f: json.dump(summary,f,indent=2)
model_hash=hashlib.sha256((MODEL_DIR/'distress_svm.pkl').read_bytes()).hexdigest()
print('Model SHA256:',model_hash)
print('Saved: /content/external_benchmark_results.csv')
print('Saved: /content/external_benchmark_summary.json')
print('Saved: /content/confusion_matrix.png')

## Interpretation

Use **balanced accuracy + Macro-F1 + distress recall + specificity + false-positive rate** together. For the strict project requirement, the key acceptance check is `FP == 0` on the designated non-distress challenge set.

Do not describe zero false positives on this finite benchmark as proof that the system can never false-trigger in the real world.